In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

In [ ]:
# Data Reading class

# Data Cleaning and Sampling class

# Hypothesis testing class

In [ ]:
# max row and columns to display
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)

# width of DataFrame
pd.set_option("display.width", 1000)

# Float number formatting
pd.options.display.float_format = "{:,.2f}".format

# p-value decimal formatting
np.set_printoptions(suppress=True)

In [ ]:
class DataReader():
  def __init__(self, file_path, sheet_name=False):
    self.file_path = file_path
    self.sheet_name = sheet_name

  # read data from Excel or CSV files
  def read_data(self):
    if self.file_path.endswith('.xlsx'):
      if self.sheet_name:
        df = pd.read_excel(self.file_path, sheet_name = self.sheet_name)
      else:
        df = pd.read_excel(self.file_path)
    else:
      df = pd.read_csv(self.file_path)

    # return the output as DataFrame
    return df

In [ ]:
cust_df = DataReader(file_path="/content/Bank_Churn_Messy.xlsx", sheet_name='Customer_Info').read_data()
cust_df.head()

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,EstimatedSalary
0,15634602,Hargrave,619,FRA,Female,42.00,2,€101348.88
1,15647311,Hill,608,Spain,Female,41.00,1,€112542.58
2,15619304,Onio,502,French,Female,42.00,8,€113931.57
3,15701354,Boni,699,FRA,Female,39.00,1,€93826.63
4,15737888,Mitchell,850,Spain,Female,43.00,2,€79084.1


In [ ]:
class DataCleaner():
  def __init__(self, df):
    self.df = df

  def choose_column(self, col_name):
    self.column = self.df[col_name]
    return self

  def clean_column(self):
    self.cleaned_column = \
    (self.column
      .str.replace('€', "")
      .astype(float))
    return self

  def choose_sample(self, n, random_state = False):
    if random_state:
      return self.cleaned_column.sample(n=n, random_state = random_state)
    else:
      return self.cleaned_column.sample(n=n)

In [ ]:
cleaner = DataCleaner(cust_df)
print(type(cleaner))
sample = (cleaner
          .choose_column('EstimatedSalary')
          .clean_column()
          .choose_sample(1000))

population = (cleaner
              .choose_column('EstimatedSalary')
              .clean_column()
              .cleaned_column)

display(sample.head())
display(population.head())

<class '__main__.DataCleaner'>


,EstimatedSalary
7041,"136,859.55"
7854,"10,096.25"
1166,"81,723.80"
5183,"156,858.20"
7638,"37,577.66"


,EstimatedSalary
0,"101,348.88"
1,"112,542.58"
2,"113,931.57"
3,"93,826.63"
4,"79,084.10"


In [ ]:
class HypothesisTesting():
  conf_level = 0.95
  alpha = round(1 - conf_level, 2)
  def __init__(self, sample, population_mean, population_std, tail = 'two'):
    assert tail in ['left', 'right', 'two'], "tail must be 'left', 'right' or 'two'"
    self.tail = tail
    self.sample = sample
    self.population_mean = population_mean
    self.population_std = population_std

  def calculate_z_score(self):
    self.z_score = ((self.sample.mean() - self.population_mean) /
                    (self.population_std / np.sqrt(len(self.sample))))
    return self

  def calculate_p_value(self):
    # calculate p-value for the choosen tail type
    if self.tail == 'left':
      self.p_value = stats.norm.cdf(self.z_score)
    elif self.tail == 'right':
      self.p_value = 1 - stats.norm.cdf(self.z_score)
    elif self.tail == 'two':
      self.p_value = 2 * (1 - stats.norm.cdf(abs(self.z_score)))
    return self

  def make_decision(self):
    if self.p_value < self.alpha:
      self.result = "Reject the null hypothesis"
    else:
      self.result = "Fail to reject the null hypothesis"
    return self

  def get_result(self):
    return self.result

In [ ]:
sample.mean()

np.float64(98832.14802000001)

In [ ]:
testing = HypothesisTesting(sample,100, population.std(), tail = 'two')

testing.calculate_z_score()
testing.calculate_p_value()
testing.make_decision()
testing.get_result()

'Reject the null hypothesis'

In [ ]:
print(testing.z_score)
print(testing.p_value)
print(testing.alpha)

51.5348371836347
0.0
0.05


In [ ]:
testing.population_std

60583.96282723315

In [ ]:
class Child(DataCleaner):
  def __init__(self, df, sample):
    super().__init__(df)
    self.sample = sample